# HUDM 5001 — Programming for Data Science
## Session 10 · Functions, Lambdas, and List Comprehensions
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cgpan/hudm5001/blob/main/Week_10/10_Functions_Lambdas_Comprehensions.ipynb)

[**Chenguang Pan**](https://cpan.ai/), [**Youmi Lab**](https://youmilab.ai/)
**Teachers College, Columbia University**


> ▶️ **How to open this notebook in Google Colab** — pick whichever is easiest:
> 1. **From Google Drive:** put this `.ipynb` in your Drive, right-click it → *Open with* → *Google Colaboratory*.
> 2. **Upload:** go to [colab.research.google.com](https://colab.research.google.com) → *File ▸ Upload notebook* → choose this file.
> 3. **From GitHub:** click the **Open in Colab** badge above (works once the notebook is pushed to the course repo).
>
> Then **run the Setup cell first** (▶ or `Shift+Enter`). Everything else runs top-to-bottom.  

About this notebook:  
This notebook is adapted from Prof. Youmi Suk's materials for HUDM 5001, Programming for Data Science ([original repository](https://github.com/youmilab/hudm5001)).  
Wording in this notebook was refined with Claude. I am responsible for its accuracy.  

Version: Nov.08.2026 12:00 pm

Welcome back to Python. Between the midterm and last week's recording on the shell and Git, it has been three weeks since our last notebook. Right before the midterm we wrote our first function, `mean_of`, and stopped there, so today we pick that thread back up. A function is a named block of code: it takes some inputs, does one job, and hands back a result. In the first half of the session you will learn the different ways to pass inputs into a function and get results back out, why a variable created inside a function is gone once the function finishes, and how to build a bigger function out of small ones. The second half adds two shortcuts you will often see in other people's code, especially in the pandas answers you find online. A `lambda` is a small function written in one line without a name, and a list comprehension builds a new list in one line. By the end of today, building a list with a loop and `.append()` should start to feel like the long way around.

---
## 1. Big Picture

### 🎯 Learning Objectives
By the end of this session you will be able to:
- Pass arguments to a function by position or by name, give a parameter a default value, and accept any number of arguments with `*args`.
- Tell printing a result apart from returning it, return several values at once, and end a function early with `return`.
- Explain scope: why a variable created inside a function disappears when the function ends, and why data should go into a function through its parameters and come out through `return`.
- Build a bigger function out of small helper functions, and write functions that other people can read.
- Write a `lambda` and hand it to `sorted()`, `max()`, `map()`, `filter()`, and the pandas `.apply()` method.
- Rewrite a loop that builds a list as a list comprehension, and build dictionaries and sets the same way.
- Decide when to use a `for` loop, a comprehension, `.apply()`, or a vectorized pandas line.

### ✅ Prerequisites
- Session 07: `for` loops, the conditional expression (`x if cond else y`), and the `mean_of` function from the end of the session.
- Session 02: lists, tuples, dictionaries, `.split()`, and the `a, b = 3, 4` unpacking trick. From the pandas sessions you need `groupby` and `value_counts()`.
- Run the Setup cell to import the libraries and set the data URL.

### 🧩 Key Concepts
`def`  `parameter / argument`  `positional argument`  `keyword argument`  `default value`  `*args`  `**kwargs`  `return`  `None`  `scope`  `local / global`  `helper function`  `predicate`  `lambda`  `sorted(key=)`  `map / filter`  `.apply()`  `list comprehension`  `dict comprehension`  `set comprehension`

### 📖 Reading
- McKinney, *Python for Data Analysis*, §3.2 (Functions) and the part of §3.1 on list, set, and dict comprehensions

### 🗺️ Today's Agenda
- 2.1 Warm-up: the function from Session 07
- 2.2 Positional and keyword arguments
- 2.3 Default values and `*args`
- 2.4 `return`: one value, several values, or none
- 2.5 Scope: local and global variables
- 2.6 Functions that call other functions
- 2.7 `lambda`: a function without a name
- 2.8 Handing a function to another function: `sorted`, `max`, `map`, `filter`, `.apply()`
- 2.9 List comprehensions
- 2.10 Dict and set comprehensions
- 2.11 Loop, comprehension, `.apply()`, or vectorize?

---
## ▶️ Setup — run this cell first
This imports the libraries and sets the web address of the Titanic passengers, who have been with us since Session 04. It also loads the `time` module for a small speed test in section 2.11.

In [ ]:
# === SETUP — run me first! ===
import numpy as np
import pandas as pd
import time     # for the speed test in section 2.11

# The Titanic passengers, read straight from the public course repo (no downloads).
TITANIC_URL = "https://raw.githubusercontent.com/cgpan/hudm5001/main/asset/titanic.csv"

print("pandas", pd.__version__, "| NumPy", np.__version__)
print("Setup complete ✅")

---
## 2. Content

### 2.1 Warm-up: the function from Session 07
This is a new notebook, so Python does not know `mean_of` yet. Run the next cell to define it again, along with the list of scores from last time.

In [ ]:
scores = [88, 92, 79, 95, 84.3]

def mean_of(values):
    """Return the arithmetic mean of a list of numbers."""
    total = 0
    for v in values:
        total += v
    return total / len(values)

print(mean_of(scores))
print(mean_of([1, 2, 3, 4]))

As a reminder, `def` starts the definition, followed by the function's name, its parameters in parentheses, and a colon. The indented lines below it are the body. The string in triple quotes is the docstring, and `return` hands the answer back to whoever called the function.

<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/function.png" width="800">
</p>

Figure source: Generated by ChatGPT.

Syntax:  
```python
def function_name(parameter_1, parameter_2):    # <---- def, a name, parameters, and a colon
    """One sentence on what the function returns."""
    your_operations                             # <---- the indented body does the work
    return result                               # <---- hands the answer back to the caller
```

Keep two moments apart. Defining a function only stores the recipe, and nothing in the body runs yet. Calling the function runs the body, with the arguments you pass plugged into the parameters. In the cell above, `values` is the parameter, while `scores` and `[1, 2, 3, 4]` are the arguments.

### 2.2 Positional and keyword arguments
`mean_of` has one parameter, so there is nothing to mix up. With two or more parameters, Python matches arguments to parameters by position: the first argument goes to the first parameter, the second to the second, and so on. Swap the order and you get a different answer, and Python will not warn you.

In [ ]:
def power(base, exp):
    """Return base raised to the power exponent."""
    return base ** exp

print(power(2, 3))     # base=2 and exp=3, <---- Directly providing the arguments when calling a function is called positional arguments.
print(power(3, 2))     # the same numbers in the other order: 3 ** 2 = 9

You can also write the parameter names in the call. These are called keyword arguments, and once the names are there, the order no longer matters. You have been using keyword arguments all semester: `normalize=True` in `value_counts()`, `ascending=False` in `sort_values()`, and `start=1` in `enumerate()`.

<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/function_2.png" width="800">
</p>

Figure source: Generated by ChatGPT.

In [ ]:
print(power(base=2, exp=3))    # <--- we call this `param = argument` style the keyword argument
print(power(exp=3, base=2))    # still 8, because the names decide which value goes where
print(power(2, exp=3))         # mixing works too, as long as the positional arguments come first

# power(base=2, 3)             # SyntaxError: positional argument follows keyword argument
# power(2)                     # TypeError: power() missing 1 required positional argument: 'exp'

Try deleting the first `#` on either of the last two lines and running the cell again. The two errors behave differently. A `SyntaxError` means Python could not read the cell at all, so nothing in it runs, not even the three lines that worked before. A `TypeError` happens while the cell is running, so the lines above it still print, and its message names the exact parameter you forgot. Put the `#` back afterwards so the notebook still runs from top to bottom.

### 2.3 Default values and `*args`
A parameter can come with a default value, which the function uses whenever the caller leaves that argument out. You have relied on defaults since the first week. `print` puts one space between the things you print because its `sep` parameter defaults to `" "`, and you can override it with a keyword argument.

In [ ]:
print("Ann", "Bob", "Cara")              # sep keeps its default: one space
print("Ann", "Bob", "Cara", sep=", ")    # override the default by name

Syntax:  
```python
def function_name(required_parameter, optional_parameter=default_value):    # <---- parameters with defaults go last
    your_operations
    return result
```

In [ ]:
def summary_line(values, ndigits=2, label="values"):
    """Return a one-line summary of a list of numbers."""
    m = mean_of(values)
    return f"{label}: mean {round(m, ndigits)} across {len(values)} items"

print(summary_line(scores))                          # both defaults used
print(summary_line(scores, ndigits=0))               # change one default
print(summary_line(scores, label="quiz scores"))     # change the other one
print(summary_line(scores, 1, "quizzes"))            # by position also works, in the order of the definition

# def broken(ndigits=2, values):    # SyntaxError: a parameter without a default cannot come after one with a default

<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/function_3.png" width="800">
</p>

Figure source: Generated by ChatGPT.

Sometimes you cannot know ahead of time how many arguments a caller will send. Put a star in front of a parameter name, as in `*args`, and Python collects all the extra positional arguments into a tuple. The star does the work, and `args` is only the usual name, so `*numbers` works just as well. A star also works the other way around inside a call: `total_of(*scores)` unpacks the list into five separate arguments.

In [ ]:
def total_of(*numbers):
    """Return the sum of however many numbers are passed in."""
    print("received:", numbers, type(numbers))    # always a tuple, even for a single number
    return sum(numbers)

print(total_of(1, 2, 3))
print(total_of(10))          # a one-item tuple prints as (10,): the comma makes it a tuple
print(total_of(*scores))     # the star unpacks the list: same as total_of(88, 92, 79, 95, 84)

In library documentation you will also run into `**kwargs`, with two stars. It collects extra keyword arguments into a dictionary. As with `*args`, the stars do the work and the name is up to you, so the example below calls it `**options`. You will rarely need to write one this semester, but it helps to recognize it when you read help pages.

In [ ]:
def show_settings(**options):
    """Print each keyword argument on its own line."""
    for name, value in options.items():
        print(f"{name} = {value}")

show_settings(learning_rate=0.01, epochs=20, model="logreg")

#### ✏️ Try It Yourself — Exercise 1
**Difficulty:** ★★★☆☆  **Skills:** default values, keyword arguments, `*args`

(a) Write `bmi(weight_kg, height_m, ndigits=1)` that returns the body-mass index (weight divided by height squared), rounded to `ndigits`. Try it with 70 kg and 1.75 m, calling it three ways: by position, with keyword arguments in the opposite order, and with `ndigits=3`.  
(b) Write `spread(*numbers)` that returns the largest number passed in minus the smallest. Check that `spread(4, 9, 2)` gives `7`, then call it on the `scores` list using `*`.

In [ ]:
# Your code here

<details>
<summary>▶ Show solution</summary>

```python
# (a)
def bmi(weight_kg, height_m, ndigits=1):
    """Return the body-mass index, rounded to ndigits."""
    return round(weight_kg / height_m ** 2, ndigits)

print(bmi(70, 1.75))                         # by position: 22.9
print(bmi(height_m=1.75, weight_kg=70))      # by name, in any order: 22.9
print(bmi(70, 1.75, ndigits=3))              # override the default: 22.857

# (b)
def spread(*numbers):
    """Return the largest number passed in minus the smallest."""
    return max(numbers) - min(numbers)

print(spread(4, 9, 2))      # 7
print(spread(*scores))      # 95 - 79 = 16
```

</details>

### 2.4 `return`: one value, several values, or none
A common first mistake is to print the answer instead of returning it. A function that prints its answer shows it on the screen, but it hands nothing back to the code that called it. A function without a `return` gives back the special value `None`, the same "nothing here" marker you used for missing readings in Session 07.

In [ ]:
def greet(name):
    """Print a greeting for name."""
    print(f"Hello, {name}!")        # shows the text, hands nothing back

result = greet("Ann")
print("result is:", result)         # None, because greet() has no return

def greeting(name):
    """Return a greeting for name."""
    return f"Hello, {name}!"        # hands the text back to the caller

message = greeting("Ann")
print(message.upper())              # now we can keep working with the result

> ⚠️ **Print is not return.** If the caller might want to use the answer, for example to store it or pass it to another function, `return` it. Use `print` when showing something is the whole job. When a function you wrote gives you `None`, a missing `return` is the first thing to check.

A function can also return several values. Separate them with commas after `return`. Python packs them into one tuple, and the unpacking trick from Session 02 splits the tuple back into separate names. Use `_` as the name for a value you do not need.

In [ ]:
def min_max_mean(values):
    """Return the smallest value, the largest value, and the mean of a list."""
    return min(values), max(values), mean_of(values)

lo, hi, avg = min_max_mean(scores)
print(lo, hi, avg)

lo, _, avg = min_max_mean(scores)    # _ is the usual name for "I don't need this one"
print(lo, avg)

print(min_max_mean(scores))          # without unpacking, you see the tuple itself

`return` also ends the function on the spot, and nothing after it runs. That makes it handy for a ladder of cases: test the first case and return, test the next one and return, and let the last line handle whatever is left. You do not need `elif` here, because each `return` already stops the function. Here is the fare ladder from Exercise 1 of Session 07, turned into a function.

In [ ]:
def fare_band(fare):
    """Return a label for a Titanic fare: first-class territory, comfortable, steerage, or bargain."""
    if fare >= 80:
        return "first-class territory"     # for a big fare, the function ends here
    if fare >= 30:
        return "comfortable"
    if fare >= 10:
        return "steerage"
    return "bargain"                       # reached only when every test above failed

for f in [7.25, 31.0, 146.52]:
    print(f, "->", fare_band(f))

#### ✏️ Try It Yourself — Exercise 2
**Difficulty:** ★★★☆☆  **Skills:** several return values, unpacking, early `return`

(a) Write `split_name(full_name)` that returns the first name and the last name as two values. The `.split()` method from Session 02 will help. Unpack `split_name("Ada Lovelace")` into two variables and print them.  
(b) Write `age_group(age)` with early returns: `"child"` under 13, `"teen"` under 20, `"adult"` under 65, and `"senior"` otherwise. Test it on `8`, `16`, `40`, and `71`.

In [ ]:
# Your code here

<details>
<summary>▶ Show solution</summary>

```python
# (a)
def split_name(full_name):
    """Return (first, last) from a "First Last" string."""
    parts = full_name.split()
    return parts[0], parts[-1]

first, last = split_name("Ada Lovelace")
print(first, "|", last)

# (b)
def age_group(age):
    """Return a life-stage label for an age."""
    if age < 13:
        return "child"
    if age < 20:
        return "teen"
    if age < 65:
        return "adult"
    return "senior"

for a in [8, 16, 40, 71]:
    print(a, "->", age_group(a))
```

</details>

### 2.5 Scope: local and global variables
At the end of Session 07 I promised to explain why `values` does not exist outside `mean_of`. That is a question about scope. A variable created inside a function, including each of its parameters, is local. Python creates it when the function is called and throws it away when the function returns. Variables created in the notebook itself, outside any function, are global. A function can read a global variable, but the notebook cannot see a function's local variables.

In [ ]:
x = 10                             # global: created in the notebook

def show():
    """Print the global x next to a local y."""
    y = 5                          # local: exists only while show() is running
    print("inside:  x =", x, "and y =", y)    # reading the global x is fine

show()
print("outside: x =", x)
# print(y)                         # NameError: y was local to show() and is gone now
# print(values)                    # NameError: the same goes for the parameter of mean_of

Assigning to a name inside a function catches many people off guard, because it creates a new local variable with that name. The global variable with the same name stays as it was. The two `count` variables below share a name and nothing else.

In [ ]:
count = 0

def bump():
    """Try to set count to 1."""
    count = 1                      # a new local variable that happens to be called count
    return count

print("bump() returns:", bump())
print("the global count is still:", count)

Writing `count += 1` inside the function fails for the same reason. The assignment makes `count` local, and the local `count` has no value yet when Python tries to add 1 to it, so you get an `UnboundLocalError`.

So how should a function update a value in the notebook? Pass the value in as an argument, return the new value, and do the assignment outside the function, where anyone reading the code can see it.

In [ ]:
def bumped(n):
    """Return n plus one. The caller decides what to do with it."""
    return n + 1

count = bumped(count)              # the update happens here, in plain sight
print("the global count is now:", count)

Python also has a `global` keyword that lets a function change a notebook variable directly. You will see it in other people's code, so here is what it looks like. I recommend that you avoid it. A function that changes things outside itself is hard to understand from its own code, and hard to reuse in another notebook.

In [ ]:
counter = 0

def bump_global():
    """Add one to the notebook's counter. It works, but please avoid this style."""
    global counter                 # from here on, counter means the notebook's counter
    counter += 1

bump_global()
bump_global()
print("counter:", counter)         # 2, although neither call shows that counter changed

#### ✏️ Try It Yourself — Exercise 3
**Difficulty:** ★★☆☆☆  **Skills:** scope, local vs. global, return and reassign

Before you run the next cell, write down what you expect the two `print` lines to show. Then run it and check.  
(a) In a comment, explain why the value printed outside the function did not change.  
(b) Fix it without `global`. Rewrite `holiday_sale(rate)` so that it takes the current rate as a parameter and returns the rate plus `0.15`, rounded to 2 decimals. Then reassign `discount` with the returned value, the way `count = bumped(count)` did above, and print `discount` to confirm it is now `0.25`.

In [ ]:
discount = 0.10

def holiday_sale():
    """Try to raise the discount to 25%."""
    discount = 0.25
    return discount

print("inside :", holiday_sale())
print("outside:", discount)

# Your code here

<details>
<summary>▶ Show solution</summary>

```python
discount = 0.10

def holiday_sale():
    """Try to raise the discount to 25%."""
    discount = 0.25          # a new local variable; the global discount is untouched
    return discount

print("inside :", holiday_sale())    # inside : 0.25
print("outside:", discount)          # outside: 0.1 (Python drops the trailing zero)

# (a) The assignment inside holiday_sale() created a local variable named discount.
#     It disappeared when the function returned, and the global discount was never touched.

# (b) pass the old value in, return the new one, and reassign outside the function
def holiday_sale(rate):
    """Return the holiday discount: the old rate plus 15 percentage points."""
    return round(rate + 0.15, 2)

discount = holiday_sale(discount)
print("outside:", discount)          # outside: 0.25
```

</details>

### 2.6 Functions that call other functions
A function can call other functions, including ones you wrote a minute ago. You have already seen this: `summary_line` in 2.3 and `min_max_mean` in 2.4 both call `mean_of`. This is how longer programs stay readable. Each small helper does one narrow job, and a bigger function puts the helpers together. In the next cell, a made-up tax rule becomes a helper, and take-home pay is one line built on top of it.

In [ ]:
def tax(amount):
    """Return the tax under a made-up rule: 0%, 15%, or 28% of the whole amount, by size."""
    if amount <= 240:
        return 0
    if amount <= 480:
        return round(amount * 0.15, 2)
    return round(amount * 0.28, 2)

def net_pay(gross):
    """Return take-home pay: the gross amount minus tax(gross)."""
    return gross - tax(gross)          # one function calling another

for gross in [200, 300, 600]:
    print(f"gross {gross}: tax {tax(gross)}, net {net_pay(gross)}")

A function that returns `True` or `False` is called a predicate. Predicates make good helpers because they give a fiddly condition a readable name. Compare `if is_vowel(ch):` with typing out `ch == "a" or ch == "e" or ch == "i" ...` every time you need the check.

In [ ]:
def is_vowel(ch):
    """Return True if the character is a vowel, in either case."""
    return ch.lower() in "aeiou"

def count_vowels(text):
    """Return the number of vowels in a string."""
    n = 0
    for ch in text:
        if is_vowel(ch):           # the helper keeps this line short and readable
            n += 1
    return n

print(count_vowels("Teachers College"))

Other people will read the functions you write, and so will you, a few months from now. Three habits make that easier:

1. Give each function one job. A name like `load_clean_plot_and_save` describes four functions squeezed into one.
2. Pick a name that says what the function returns. You can guess what `mean_of`, `is_vowel`, and `fare_band` give back from their names alone, while `helper2` tells the reader nothing.
3. Write a docstring every time. One sentence is enough.

#### ✏️ Try It Yourself — Exercise 4
**Difficulty:** ★★★★☆  **Skills:** predicate helper, one function calling another, loop with a counter

(a) Write a predicate `is_long_word(word, min_len=6)` that returns `True` when `len(word) >= min_len`.  
(b) Write `count_long_words(text, min_len=6)` that splits the text into words and calls `is_long_word(word, min_len)` inside a loop to count them. Test it on `sentence` with the default, then again with `min_len=4`.

In [ ]:
sentence = "functions turn repeated code into reusable tools"

# Your code here

<details>
<summary>▶ Show solution</summary>

```python
def is_long_word(word, min_len=6):
    """Return True if the word has at least min_len characters."""
    return len(word) >= min_len

def count_long_words(text, min_len=6):
    """Return how many words in text pass is_long_word()."""
    n = 0
    for word in text.split():
        if is_long_word(word, min_len):
            n += 1
    return n

sentence = "functions turn repeated code into reusable tools"
print(count_long_words(sentence))              # 3: functions, repeated, reusable
print(count_long_words(sentence, min_len=4))   # 7: every word has at least 4 letters
```

</details>

### 2.7 `lambda`: a function without a name
Some functions are so small that a `def`, a name, and a docstring feel like more ceremony than the job needs. For those, Python has `lambda`. A lambda is a one-line function without a name. You write the keyword `lambda`, the parameters, a colon, and a single expression. Python returns the value of that expression automatically, so there is no `return` keyword.

<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/lambda.gif" width="800">
</p>

Figure source: [runestone](https://runestone.academy/ns/books/published/fopp/AdvancedFunctions/Anonymousfunctionswithlambdaexpressions.html)

Syntax:  
```python
lambda parameter_1, parameter_2: expression     # <---- no def, no name, no return keyword
```

In [ ]:
square = lambda x: x ** 2           # parameter, colon, one expression
print(square(7))

def square_def(x):                  # the same function, written with def
    """Return x squared."""
    return x ** 2
print(square_def(7))

print((lambda x: x ** 2)(7))        # a lambda is a value, so you can even call it on the spot

add = lambda a, b: a + b            # more than one parameter is fine
print(add(3, 4))

is_adult = lambda age: age >= 18    # a lambda can be a predicate, too
print(is_adult(16), is_adult(30))

A lambda holds exactly one expression, so loops, `if` statements, and docstrings do not fit inside it. The conditional expression from Session 07 does fit, because it is an expression.

In [ ]:
grade = lambda score: "pass" if score >= 60 else "fail"
print(grade(72), grade(41))

If you want more than one line, write a `def`. You will also rarely see a lambda stored under a name the way `square` is above. Python's style guide, PEP 8, asks you to use `def` whenever you want a function with a name. I named the lambdas here only so that we could call them. In practice, a lambda is most useful as a small rule you hand to another function, which is the next section.

### 2.8 Handing a function to another function
Some functions take another function as an argument: a small rule that tells them how to do their job.

**`sorted(..., key=...)`.** By default, `sorted` compares the items themselves. When you pass a `key` function, `sorted` calls it on each item and sorts by what it returns. That is how you sort words by length, or records by one of their fields.

In [ ]:
words = ["pandas", "if", "lambda", "def", "comprehension"]
print(sorted(words))                              # alphabetical
print(sorted(words, key=len))                     # by length: len is a function, so it works as a key
print(sorted(words, key=len, reverse=True))       # longest first

students = [("Ann", 88), ("Bob", 92), ("Cara", 79)]
print(sorted(students, key=lambda s: s[1], reverse=True))   # by score, highest first

`key=` works the same way in `max()` and `min()`, which turns the best-so-far loop from Session 07 into a single line. `max` hands back the whole winning item, so you get the name along with the score.

In [ ]:
top = max(students, key=lambda s: s[1])
print("top student:", top[0], "with", top[1])

**`map` and `filter`.** `map(f, xs)` calls `f` on every item of `xs`, and `filter(f, xs)` keeps the items for which `f` returns `True`. Both hand back a one-time object, called an iterator, that produces its values only when asked. Wrap it in `list()` to see the results, the same way we printed `list(zip(names, scores))` in Session 07. `list()` uses the iterator up, so asking a second time gives an empty list.

In [ ]:
doubled = map(lambda s: s * 2, scores)
print(doubled)                     # a map object: the values are not computed yet
print(list(doubled))               # list() asks for all of them
print(list(doubled))               # [] because the iterator is used up

high = filter(lambda s: s > 85, scores)
print(list(high))

**pandas `.apply()`.** Call `.apply(f)` on a Series, and pandas runs `f` on each value and returns a new Series with the results. It is an easy way to label a column with a rule you wrote yourself. Any function works, either a lambda or a `def` you already have. First, load the passengers.

In [ ]:
titanic = pd.read_csv(TITANIC_URL)

# a lambda for a quick one-off rule
titanic["pricey"] = titanic["fare"].apply(lambda f: "yes" if f > 50 else "no")
print(titanic["pricey"].value_counts())

# a def we already have, for the four-way ladder
titanic["band"] = titanic["fare"].apply(fare_band)
print(titanic["band"].value_counts())

Look at how `fare_band` is passed: without parentheses. `fare_band` is the function itself, and `.apply()` calls it once per row. If you write `.apply(fare_band())`, Python tries to call `fare_band` right away, with no fare, and stops with `TypeError: fare_band() missing 1 required positional argument: 'fare'`.

> ⚠️ **`.apply()` is still a loop.** pandas calls your function once per value, in plain Python. That is fine for labels and text rules. For arithmetic, such as doubling a column or adding two columns together, use a vectorized line like `titanic["fare"] * 2`, which is much faster. You will time the difference in 2.11.

#### ✏️ Try It Yourself — Exercise 5
**Difficulty:** ★★★★☆  **Skills:** `lambda`, `sorted(key=)`, `max(key=)`, `filter`, pandas `.apply()`

The list in the next cell holds four programs and their enrollments.  
(a) Sort the programs by enrollment, largest first, using `sorted` with a `lambda` key.  
(b) Use `max(..., key=...)` to find the program with the most students, and print only its name.  
(c) Use `filter` and a `lambda` to keep the programs with at least 30 students, then loop over the result and print each program's name and enrollment.  
(d) On `titanic`, use `.apply` with a `lambda` to make a column `pclass_label` that turns `1`, `2`, `3` into `"first"`, `"second"`, `"third"`. A dictionary lookup inside the lambda works well. Print the new column's `value_counts()`.

In [ ]:
programs = [("Applied Stats", 32), ("Learning Analytics", 28), ("Measurement", 24), ("Cognitive Science", 36)]

# Your code here

<details>
<summary>▶ Show solution</summary>

```python
programs = [("Applied Stats", 32), ("Learning Analytics", 28), ("Measurement", 24), ("Cognitive Science", 36)]

# (a)
print(sorted(programs, key=lambda p: p[1], reverse=True))

# (b)
biggest = max(programs, key=lambda p: p[1])
print("most students:", biggest[0])

# (c)
big = list(filter(lambda p: p[1] >= 30, programs))
for name, n in big:
    print(name, "has", n, "students")

# (d)
labels = {1: "first", 2: "second", 3: "third"}
titanic["pclass_label"] = titanic["pclass"].apply(lambda c: labels[c])
print(titanic["pclass_label"].value_counts())
```

</details>

### 2.9 List comprehensions
You know the long way to build a list: start with an empty list, loop, and `.append()` one item on each pass. Python has a one-line form for exactly this pattern, called a list comprehension. Read it from left to right as "this expression, for each item in that iterable."

In [ ]:
# the long way: empty list, loop, append
squares = []
for n in range(1, 6):
    squares.append(n ** 2)
print(squares)

# the comprehension: the same list in one line
squares = [n ** 2 for n in range(1, 6)]
print(squares)

Syntax:  
```python
new_list = [expression for item in iterable]                  # <---- transform every item
new_list = [expression for item in iterable if condition]     # <---- keep only the items that pass the test
```

The expression can be anything that produces a value: arithmetic, a method call, or a function you wrote. Add an `if` at the end and the comprehension also filters, so only the items that pass the test make it into the new list.

In [ ]:
words = ["pandas", "if", "lambda", "def", "comprehension"]

print([w.upper() for w in words])                    # change every item
print([w for w in words if len(w) > 3])              # keep some of the items
print([len(w) for w in words if w != "if"])          # filter first, then change
print([fare_band(f) for f in [7.25, 31.0, 146.52]])  # your own function as the expression

The word `if` can sit in two places in a comprehension, and the two places do different jobs:
- An `if` at the end, after the `for`, is a filter. It decides which items get in, so fewer items can come out than went in. It cannot have an `else`.
- An `if ... else` at the front, before the `for`, is the conditional expression from Session 07. Every item gets in, and the expression decides what value each one becomes. Here the `else` is required.

In [ ]:
print([s for s in scores if s >= 85])                     # filter: 3 of the 5 scores come out
print(["high" if s >= 85 else "low" for s in scores])     # pick a value: all 5 come out

Comprehensions are handy in data work. Here are three one-liners on the Titanic table. The second one is the missing-value check from Session 07, now in a single line.

In [ ]:
# column names that contain the letter "p"
print([col for col in titanic.columns if "p" in col])

# the missing-value check from Session 07, as a comprehension
print([col for col in titanic.columns if titanic[col].isna().sum() > 0])

# labels for the first five fares
print([fare_band(f) for f in titanic["fare"].head()])

A comprehension can stand in for most uses of `map` and `filter`. `[f(x) for x in xs]` gives the same list as `list(map(f, xs))`, and most people find the comprehension easier to read. Keep comprehensions short, with one transformation and at most one filter. When you need several steps, or want to print along the way to see what is going on, go back to a regular `for` loop.

#### ✏️ Try It Yourself — Exercise 6
**Difficulty:** ★★★☆☆  **Skills:** list comprehension, filter condition, conditional expression

Write each of these as a single list comprehension.  
(a) The cubes of the numbers 1 through 8.  
(b) From `review_words`, the words that are not in `stop_words`. In Session 02 you removed stop words with `.replace()`; this time one comprehension does the job.  
(c) From `ages`, the label `"minor"` or `"adult"` for each age, using a conditional expression with 18 as the cutoff.  
(d) From `titanic`, the names of the numeric columns, meaning the ones whose dtype is `int64` or `float64`. Test each column with `titanic[col].dtype in ["int64", "float64"]`.

In [ ]:
review_words = ["i", "am", "not", "a", "fan", "of", "the", "film"]
stop_words = ["a", "am", "an", "i", "the", "of"]
ages = [8, 16, 40, 71]

# Your code here

<details>
<summary>▶ Show solution</summary>

```python
# (a)
print([n ** 3 for n in range(1, 9)])

# (b)
review_words = ["i", "am", "not", "a", "fan", "of", "the", "film"]
stop_words = ["a", "am", "an", "i", "the", "of"]
print([w for w in review_words if w not in stop_words])    # ['not', 'fan', 'film']

# (c)
ages = [8, 16, 40, 71]
print(["adult" if a >= 18 else "minor" for a in ages])

# (d)
print([col for col in titanic.columns if titanic[col].dtype in ["int64", "float64"]])
# passenger_id passes the test too: it is stored as a number, even though it is only a row label
```

</details>

### 2.10 Dict and set comprehensions
The same idea builds dictionaries. Use curly braces instead of square brackets, and write the expression as `key: value`. A common source is a loop over `.items()`, which means any `groupby` summary can become a plain dictionary in one line.

Syntax:  
```python
new_dict = {key_expression: value_expression for item in iterable}
new_dict = {k: v for k, v in old_dict.items() if condition}      # <---- .items() hands over each key and its value
```

In [ ]:
names = ["Ann", "Bob", "Cara", "Dan", "Eve"]

print({name: len(name) for name in names})            # each name and its number of letters
print({name: s for name, s in zip(names, scores)})     # two lists become one lookup table

# a groupby summary as a plain dictionary, rounded as it is built
print({pclass: round(m, 2) for pclass, m in titanic.groupby("pclass")["fare"].mean().items()})

Session 07 got the same fare dictionary with `.round(2).to_dict()`, and for this exact job that works just as well. A comprehension pays off when you want to filter or reshape along the way. Add an `if` to keep only part of a dictionary, or swap the key and the value to turn a lookup table around. Text analysis often needs the second move, because a model usually wants one lookup from words to numbers and another from numbers back to words.

In [ ]:
settings = {"max_depth": 5, "n_trees": 100, "max_iter": 20, "seed": 42}
print({k: v for k, v in settings.items() if "max" in k})       # keep the settings with "max" in the name

letter_to_idx = {"a": 0, "b": 1, "c": 2}
print({idx: letter for letter, idx in letter_to_idx.items()})  # the reverse lookup

In Assignment 05 you wrote `audit(df)`: a loop over the columns that skipped the clean ones with `continue` and stored each missing count in a dictionary as a plain `int`. A dict comprehension does the same job in one expression.

In [ ]:
def audit(df):
    """Return {column: number of missing values} for every column that has at least one."""
    # int() turns the NumPy integer from .sum() into a plain Python int, as in Assignment 05
    return {col: int(df[col].isna().sum()) for col in df.columns if df[col].isna().sum() > 0}

print(audit(titanic))

Curly braces without the colon give a set comprehension. It is a quick way to collect the distinct values of something, because a set keeps each value only once. A set has no order, so the items can print in any order.

In [ ]:
words = ["pandas", "if", "lambda", "def", "comprehension"]
print({w[0] for w in words})                       # the distinct first letters
print({fare_band(f) for f in titanic["fare"]})     # every fare band that occurs in the data

#### ✏️ Try It Yourself — Exercise 7
**Difficulty:** ★★★★☆  **Skills:** dict comprehension, `.items()`, filter condition, `groupby`

(a) Build a dictionary that maps each `pclass` to its survival rate (the mean of `survived`), rounded to 3 decimals, with a dict comprehension over the `.items()` of a `groupby` result.  
(b) From that dictionary, build a second one that keeps only the classes whose rate is above the ship-wide rate, `titanic["survived"].mean()`.  
(c) Build a dictionary that maps each numeric column name to its mean, rounded to 2. Reuse the dtype test from Exercise 6(d), and wrap each mean in `float(...)` before rounding. `.mean()` gives back a NumPy number, which shows up as `np.float64(29.7)` in a printed dictionary, while the `.items()` in (a) already hands you plain Python numbers.

In [ ]:
# Your code here

<details>
<summary>▶ Show solution</summary>

```python
# (a)
survival = {pclass: round(rate, 3)
            for pclass, rate in titanic.groupby("pclass")["survived"].mean().items()}
print(survival)                                   # {1: 0.63, 2: 0.473, 3: 0.242}

# (b)
overall = titanic["survived"].mean()
print({pclass: rate for pclass, rate in survival.items() if rate > overall})   # first and second class

# (c)
print({col: round(float(titanic[col].mean()), 2)
       for col in titanic.columns if titanic[col].dtype in ["int64", "float64"]})
# the passenger_id mean (446.0) means nothing, since the id is only a row label
```

</details>

### 2.11 Loop, comprehension, `.apply()`, or vectorize?
You now have several ways to do something to every item, and `def` to package any of them. Here is how I would choose. For arithmetic on a column, speed decides. For everything else, pick the version that reads best.

| You want to... | Reach for | Why |
|---|---|---|
| do arithmetic on a whole column | a vectorized pandas or NumPy line (`df["x"] * 2`) | by far the fastest, and usually the shortest |
| build a new list with one transformation and at most one filter | a comprehension | one line that says what it builds |
| turn a column into labels with a rule | `.apply()` with a lambda or a `def` | the rule stays readable, and pandas handles the looping |
| do several steps, print as you go, stop early, or write files | a `for` loop | a comprehension cannot `break`, and printing inside one is awkward |
| use the same logic in two places | a `def` | write it once and use it wherever you need it |

A comprehension is still a loop underneath. Python runs it one item at a time, so it reads better than the loop it replaces but runs at about the same speed. The vectorized line hands the whole column to NumPy, which does the work in compiled code. All four versions below double every Titanic fare and give the same numbers.

In [ ]:
fares = titanic["fare"]

by_vector = fares * 2                       # numbers, so this is the first choice
by_comp   = [f * 2 for f in fares]          # fine for lists, slower on big columns
by_apply  = fares.apply(lambda f: f * 2)    # works, but .apply() is meant for rules and labels
by_loop   = []                              # the long way
for f in fares:
    by_loop.append(f * 2)

print(list(by_vector.head(3)), by_comp[:3], list(by_apply.head(3)), by_loop[:3])

With 891 rows, all four finish almost instantly. To see a real difference, the next cell makes a column of a million random numbers and times each version on it. The helper `seconds_to_run` takes a function, runs it once, and returns how long that took. Each `lambda:` with nothing between `lambda` and the colon is a function with no parameters. It wraps one line of code so that `seconds_to_run` can decide when to run it. The loop needs more than one line, so it gets a small `def` instead, passed without parentheses just like `fare_band` in 2.8.

In [ ]:
np.random.seed(5001)
big = pd.Series(np.random.uniform(0, 500, 1000000))    # a million made-up fares

def seconds_to_run(job):
    """Run job() once and return how many seconds it took, rounded to 4 decimals."""
    start = time.perf_counter()
    job()
    return round(time.perf_counter() - start, 4)

def double_with_loop():
    """Double every value in big the long way: empty list, loop, append."""
    out = []
    for x in big:
        out.append(x * 2)
    return out

print("vectorized:   ", seconds_to_run(lambda: big * 2))
print("comprehension:", seconds_to_run(lambda: [x * 2 for x in big]))
print("for loop:     ", seconds_to_run(double_with_loop))     # a def, passed without parentheses
print(".apply():     ", seconds_to_run(lambda: big.apply(lambda x: x * 2)))

Run the cell a couple of times. The exact numbers depend on the computer, but the vectorized line should win by a wide margin, while the comprehension and the loop land close to each other. The rule from Session 07 still holds: let pandas and NumPy do the arithmetic.

---
## 3. Conclusions & Key Takeaways
- Python matches arguments to parameters by position unless you name them. Keyword arguments can come in any order, as long as they come after the positional ones.
- A default value makes a parameter optional, and parameters with defaults go after the ones without. `*args` collects any number of extra positional arguments into a tuple, and a star in a call, as in `f(*my_list)`, unpacks a list into separate arguments.
- `print` shows a value, and `return` hands it back to the caller. A function without a `return` gives back `None`. To return several values, separate them with commas and unpack them on the other side. A `return` ends the function right away.
- Variables created inside a function, parameters included, are local and disappear when the function ends. Assigning to a name inside a function creates a new local variable and leaves the global one alone. Pass values in as arguments, hand results back with `return`, and avoid `global`.
- Small helpers, especially predicates that return `True` or `False`, keep bigger functions short. Give each function one job, a name that says what it returns, and a docstring.
- A `lambda` is a one-expression function without a name. It is most useful as the argument to `sorted(key=)`, `max(key=)`, `map`, `filter`, and pandas `.apply()`. When you want a function with a name, write a `def`.
- A list comprehension, `[expr for x in xs if cond]`, replaces the empty-list, loop, and `.append()` pattern. An `if` at the end filters the items, and an `if ... else` at the front picks a value for every item. Dict comprehensions (`{k: v for ...}`) and set comprehensions (`{expr for ...}`) work the same way.
- Use vectorized pandas or NumPy for arithmetic, `.apply()` for labels and rules, comprehensions for short lists, and a `for` loop when you need several steps. Anything you use twice belongs in a `def`.

### 🧾 Quick Reference
| Task | Code |
|---|---|
| define and call | `def f(a, b=1):` … `return x`, then `f(3)` or `f(3, b=2)` |
| keyword arguments | `f(b=2, a=3)` |
| any number of arguments | `def f(*args):` (a tuple inside), and `f(*my_list)` in a call |
| several return values | `return lo, hi`, then `lo, hi = f(x)` |
| skip a returned value | `lo, _ = f(x)` |
| docstring | first line inside `def`: `"""What it returns."""`, shown by `help(f)` |
| lambda | `lambda x: x ** 2` |
| sort by a rule | `sorted(items, key=lambda t: t[1], reverse=True)` |
| best item by a rule | `max(items, key=lambda t: t[1])` |
| map and filter | `list(map(f, xs))`, `list(filter(pred, xs))` |
| label a column | `df["label"] = df["x"].apply(rule)` |
| list comprehension | `[f(x) for x in xs if cond]` |
| pick a value for each item | `["yes" if cond else "no" for x in xs]` |
| dict comprehension | `{k: v for k, v in d.items() if cond}` |
| groupby to dict | `{g: round(m, 2) for g, m in df.groupby("g")["x"].mean().items()}` |
| set comprehension | `{f(x) for x in xs}` |

### ⏭️ Coming Up Next
**Classes.** A function packages behavior. A class packages behavior together with the data it works on, into a new type of your own. A DataFrame is one: it carries its table, and it also knows how to compute `.mean()` on that table. Next week you will see what `self` and `__init__` mean, and why every pandas object you have used this semester comes from a class. (The syllabus listed classes for Week 10, so we are one week behind the printed schedule.) Reading: the [classes chapter](https://docs.python.org/3/tutorial/classes.html) of the official Python tutorial.

### 📌 This Week's Assignment
An assignment on functions, lambdas, and comprehensions. I will post it on Canvas and in the `Assignments/` folder of the course GitHub.

> 💡 **Tip:** When a function gives you a strange answer, call it on the smallest input you can think of, such as a list of two numbers, and print what comes back. If you see `None`, look for a missing `return`. And when you catch yourself copying the same block of code a second time, turn it into a function. In 2.11, `seconds_to_run` saved us from writing the same timing lines four times.